# MalPDFDeepDetection — Cloud Hybrid CPU+GPU Bulk Extraction

Recommended Workflow:
- **Phase 1 (`CPU · Xxsmall` — 4 CPU / 16 GB RAM):** Acquire & unpack `~20 GB` verified datasets (`--acquire --device cpu`) with zero GPU spend.
- **Phase 2 (`GPU · Xsmall` — 8 CPU / 32 GB RAM / 1 NVIDIA L4 GPU):** Run `--extract` with CPU isolated parsers (`45s wall / 35s CPU`, `coalesce_checkpoints=True`) + CUDA batched `256x256` Markov & `23D` RWSE projection.
- **Hygiene & Checkpoints:** Filters `.json` non-PDFs and non-dataset assets, deduplicates by SHA-256 with strict label-provenance priority, and checkpoints every `500`-PDF chunk under `outputs/bulk_extract/`.

In [ ]:
!nvidia-smi || true
!nproc
!free -h
!df -h .

In [ ]:
import os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
VENV_PY = ROOT / ".venv/bin/python"
PY = str(VENV_PY if VENV_PY.is_file() else sys.executable)
if not VENV_PY.is_file():
    subprocess.check_call([PY, "-m", "pip", "install", "-q", "-e", ".[dev]", "py7zr"])
print("Working root:", ROOT, "| Python:", PY)
subprocess.check_call([PY, "scripts/cloud_bulk_extract.py", "--benchmark-only"])

In [ ]:
# Step 1: Pin & verify Arlington TSV model (commit b7d880b7)
subprocess.check_call([PY, "scripts/cloud_bulk_extract.py", "--execute-cloud-extraction", "--setup-arlington"])

In [ ]:
# Phase 1 (Run on CPU · Xxsmall): Acquire verified datasets into outputs/ephemeral_raw/
subprocess.check_call([
    PY, "-u", "scripts/cloud_bulk_extract.py",
    "--execute-cloud-extraction",
    "--acquire",
    "--raw-dir", "outputs/ephemeral_raw",
    "--device", "cpu",
    "--download-workers", "2",
    "--hard-cap-gb", "20.0",
])

In [ ]:
# Phase 2 (Switch compute to GPU · Xsmall): Stream Hybrid CPU+GPU extraction by 500-PDF chunk
# Optional: add "--prune-raw-after-chunk" if disk space is tight.
cmd = [
    PY, "-u", "scripts/cloud_bulk_extract.py",
    "--execute-cloud-extraction",
    "--extract",
    "--raw-dir", "outputs/ephemeral_raw",
    "--output-dir", "outputs/bulk_extract",
    "--device", "auto",
    "--chunk-size", "500",
    "--wall-seconds", "45",
    "--cpu-seconds", "35",
]
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"Extraction exited with code {proc.returncode}")

In [ ]:
import json
from pathlib import Path

summary = json.loads(Path("outputs/bulk_extract/final/summary.json").read_text())
print(json.dumps(summary, indent=2))